# Corrected main prefix comparison

Initial scope: **IPv4 overall traffic**, **Raw** (no scan removal), **native** selected-prefix membership, and **all** selected prefixes. Canonical `src_ip` and `dst_ip` retain first-observed direction; they are not initiator/responder roles.

In [ ]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from mawi_global_analysis.io import load_run

root = Path(os.environ.get('MAWI_ANALYSIS_ROOT', '.')).resolve()
dataset_id = os.environ.get('MAWI_DATASET_ID', 'fixture')
run_name = os.environ.get('MAWI_RUN_NAME', 'baseline')
run = load_run(dataset_id, run_name, root=root)
provenance = pd.DataFrame([{'dataset': dataset_id, 'run_name': run_name, 'analysis_scope': 'IPv4 overall / Raw / native / all selected prefixes', 'config_hash': run.manifest.get('config', {}).get('hash'), 'input_sha256': run.manifest.get('input', {}).get('sha256'), 'git_commit': run.manifest.get('git_commit')}])
display(provenance)

## Canonical-table derivation

All views below are derived here from canonical `flows.csv`, run-local neutral `flow_labels.csv`, the prefix ledger, and `flow_prefix_membership.csv`; no plot-specific pipeline data is read. Raw analysis retains every IPv4 flow regardless of neutral label columns.

In [ ]:
# Raw means join labels for provenance but do not filter any flow.
flows = run.flows.loc[pd.to_numeric(run.flows['ip_version']) == 4].copy()
raw_flows = flows.merge(run.labels, on='flow_id', how='left', validate='one_to_one')
selected_prefixes = run.prefixes.loc[run.prefixes['selected_for_analysis'] == True, ['prefix', 'prefix_length', 'seen_as_src_prefix', 'seen_as_dst_prefix']].copy()
native_membership = run.membership.loc[run.membership['analysis_scope'] == 'native'].copy()
prefix_flows = native_membership.merge(raw_flows, on='flow_id', how='inner', validate='many_to_one')
selected_scope_flows = raw_flows.loc[raw_flows['flow_id'].isin(native_membership['flow_id'].unique())]
display(pd.DataFrame([{'ipv4_raw_flow_count': len(raw_flows), 'neutral_strict_removed_labels': int(raw_flows['strict_removed'].fillna(False).sum()), 'neutral_broad_removed_labels': int(raw_flows['broad_removed'].fillna(False).sum()), 'selected_native_prefix_count': len(selected_prefixes), 'native_membership_rows': len(native_membership), 'unique_flows_matching_any_native_prefix': native_membership['flow_id'].nunique()}]))
display(selected_prefixes.sort_values('prefix').head(20))

## TCP/UDP packet-count flow-length distributions

Flow length here is canonical per-flow `packet_count`. The ECDF and CCDF retain the distribution and tail information beyond the median.

In [ ]:
def ecdf(values):
    ordered = np.sort(np.asarray(values, dtype=float))
    return ordered, np.arange(1, len(ordered) + 1) / len(ordered)

protocols = {'TCP': ['6', '6.0', 'tcp'], 'UDP': ['17', '17.0', 'udp']}
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for protocol, values in protocols.items():
    for scope, frame, style in [('IPv4 overall', raw_flows, '-'), ('Native-prefix associated', selected_scope_flows, '--')]:
        packets = frame.loc[frame['protocol'].astype(str).isin(values), 'packet_count']
        if packets.empty:
            continue
        x, y = ecdf(packets)
        label = f'{protocol}: {scope}'
        axes[0].hist(packets, bins='auto', histtype='step', linewidth=2, linestyle=style, label=label)
        axes[1].step(x, y, where='post', linestyle=style, label=label)
        axes[2].step(x, 1 - y, where='post', linestyle=style, label=label)
axes[0].set(title='Packet-count histogram', xlabel='Packets per flow', ylabel='Flow count', xscale='log', yscale='log')
axes[1].set(title='Packet-count ECDF', xlabel='Packets per flow', ylabel='Cumulative probability', xscale='log')
axes[2].set(title='Packet-count CCDF', xlabel='Packets per flow', ylabel='Tail probability', xscale='log', yscale='log')
for axis in axes:
    axis.legend()
fig.tight_layout()

In [ ]:
# One row for every selected native prefix, including prefixes with zero canonical membership.
prefix_metrics = prefix_flows.groupby('analysis_prefix', as_index=False).agg(flow_count=('flow_id', 'size'), packet_count=('packet_count', 'sum'), frame_byte_count=('frame_byte_count', 'sum'), ip_byte_count=('ip_byte_count', 'sum'), median_packet_count=('packet_count', 'median'), median_frame_byte_count=('frame_byte_count', 'median'), median_duration=('duration', 'median'), q90_duration=('duration', lambda values: values.quantile(0.9)), q99_duration=('duration', lambda values: values.quantile(0.99)))
prefix_summary = selected_prefixes.rename(columns={'prefix': 'analysis_prefix'}).merge(prefix_metrics, on='analysis_prefix', how='left', validate='one_to_one')
zero_volume_prefixes = prefix_summary['flow_count'].isna()
prefix_summary[['flow_count', 'packet_count', 'frame_byte_count', 'ip_byte_count']] = prefix_summary[['flow_count', 'packet_count', 'frame_byte_count', 'ip_byte_count']].fillna(0)
display(pd.DataFrame([{'selected_prefixes_with_zero_canonical_membership': int(zero_volume_prefixes.sum()), 'selected_prefix_count': len(prefix_summary)}]))
display(prefix_summary.sort_values('frame_byte_count', ascending=False).head(20))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
positive_volume = prefix_summary.loc[prefix_summary['frame_byte_count'] > 0]
for axis, y_column, title in zip(axes, ['median_packet_count', 'median_frame_byte_count', 'median_duration'], ['Traffic volume vs median packets', 'Traffic volume vs median frame bytes', 'Traffic volume vs median duration']):
    axis.scatter(positive_volume['frame_byte_count'], positive_volume[y_column], alpha=0.75)
    axis.set(xscale='log', yscale='log' if (positive_volume[y_column] > 0).all() else 'linear', xlabel='Native-prefix frame bytes', ylabel=y_column, title=title)
fig.tight_layout()

In [ ]:
# The selected-prefix series is deduplicated by flow_id for this combined-scope view.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for label, values in [('IPv4 overall (Raw)', raw_flows['duration']), ('Any selected native prefix (Raw)', selected_scope_flows['duration'])]:
    x, y = ecdf(values)
    axes[0].step(x, y, where='post', label=label)
    axes[1].step(x, 1 - y, where='post', label=label)
axes[0].set(title='Flow-duration ECDF', xlabel='Duration (seconds)', ylabel='Cumulative probability')
axes[1].set(title='Flow-duration CCDF', xlabel='Duration (seconds)', ylabel='Tail probability', yscale='log')
for axis in axes:
    axis.legend()
fig.tight_layout()

This corrected baseline intentionally uses all selected non-overlapping native IPv4 prefixes. Prefix membership is `src_ip ∈ prefix OR dst_ip ∈ prefix`; `src_match` and `dst_match` remain observation-direction facts, not traffic-role labels.